In [1]:
import pandas as pd

df = pd.read_csv("hotel_bookings.csv.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(df.head())


Dataset loaded successfully!
Shape: (119390, 32)
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0          

In [2]:
import pandas as pd
df = pd.read_csv(
    "hotel_bookings.csv.csv",
    parse_dates=["reservation_status_date"]
)
before = df.memory_usage(deep=True).sum() / 1024**2
print("Memory before:", round(before, 2), "MB")


for col in df.select_dtypes(include="number").columns:
       df[col] = pd.to_numeric(df[col], downcast="integer")

for col in df.select_dtypes(include="object").columns:
    if df[col].nunique() < 50:
     df[col] = df[col].astype("category")

after = df.memory_usage(deep=True).sum() / 1024**2
saving = (before - after) / before * 100

print("Memory after:", round(after, 2), "MB")
print("Memory saved:", round(saving, 2), "%")

/tmp/ipykernel_57552/1891043692.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


Memory before: 88.1 MB
Memory after: 13.77 MB
Memory saved: 84.37 %


In [3]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(),
    "most_frequent": df.mode().iloc[0],
    "frequency": df.apply(lambda x: x.value_counts().iloc[0])
})

print(profile)

                                         dtype  missing_count  \
hotel                                 category              0   
is_canceled                               int8              0   
lead_time                                int16              0   
arrival_date_year                        int16              0   
arrival_date_month                    category              0   
arrival_date_week_number                  int8              0   
arrival_date_day_of_month                 int8              0   
stays_in_weekend_nights                   int8              0   
stays_in_week_nights                      int8              0   
adults                                    int8              0   
children                               float64              4   
babies                                    int8              0   
meal                                  category              0   
country                                 object            488   
market_segment           

In [4]:
duplicates = df.duplicated().sum()

keys = ["hotel", "arrival_date_year", "arrival_date_month",
        "arrival_date_day_of_month", "adults"]

key_duplicates = df.duplicated(subset=keys).sum()

print("Exact duplicate rows:", duplicates)
print("Duplicate rows on keys:", key_duplicates)
print("Rule: Remove exact duplicates, but keep key duplicates because they can be valid bookings.")

Exact duplicate rows: 31994
Duplicate rows on keys: 114721
Rule: Remove exact duplicates, but keep key duplicates because they can be valid bookings.


In [5]:
print("Before:")
print(df["hotel"].value_counts())

df["hotel"] = df["hotel"].astype(str).str.strip().str.lower()

df["hotel"] = df["hotel"].replace({
    "resort hotel": "Resort Hotel",
    "city hotel": "City Hotel"
})

print("\nAfter:")
print(df["hotel"].value_counts())

Before:
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64

After:
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64


In [6]:
import pandas as pd

text = df["reservation_status_date"].astype(str)

df["number"] = text.str.extract(r"(\d+)", expand=False)

df["number"] = pd.to_numeric(df["number"], errors="coerce")

print("Rows that did not match:", df["number"].isna().sum())
print(df[["reservation_status_date", "number"]].head())

Rows that did not match: 0
  reservation_status_date  number
0              2015-01-07    2015
1              2015-01-07    2015
2              2015-02-07    2015
3              2015-02-07    2015
4              2015-03-07    2015


In [7]:
df["reservation_status_date"] = pd.to_datetime(
    df["reservation_status_date"],
    errors="coerce",
    dayfirst=True
)

print("Failed dates:", df["reservation_status_date"].isna().sum())

df["year"] = df["reservation_status_date"].dt.year
df["month"] = df["reservation_status_date"].dt.month
df["day_of_week"] = df["reservation_status_date"].dt.dayofweek
df["quarter"] = df["reservation_status_date"].dt.quarter

print(df[[
    "reservation_status_date",
    "year",
    "month",
    "day_of_week",
    "quarter"
]].head())

Failed dates: 0
  reservation_status_date  year  month  day_of_week  quarter
0              2015-01-07  2015      1            2        1
1              2015-01-07  2015      1            2        1
2              2015-02-07  2015      2            5        1
3              2015-02-07  2015      2            5        1
4              2015-03-07  2015      3            5        1


In [8]:
def average_adr(x):
    return x.mean()

summary = df.groupby("hotel").agg(
    booking_count=("hotel", "count"),
    average_adr=("adr", "mean"),
    highest_adr=("adr", "max"),
    average_lead_time=("lead_time", "mean"),
    custom_average=("adr", average_adr)
)

print(summary)

              booking_count  average_adr  highest_adr  average_lead_time  \
hotel                                                                      
City Hotel            79330   105.304465       5400.0         109.735724   
Resort Hotel          40060    94.952930        508.0          92.675686   

              custom_average  
hotel                         
City Hotel        105.304465  
Resort Hotel       94.952930  


In [9]:
df["adr_minus_group_mean"] = (
    df["adr"] - df.groupby("hotel")["adr"].transform("mean")
)

print(df[["hotel", "adr", "adr_minus_group_mean"]].head())

          hotel   adr  adr_minus_group_mean
0  Resort Hotel   0.0             -94.95293
1  Resort Hotel   0.0             -94.95293
2  Resort Hotel  75.0             -19.95293
3  Resort Hotel  75.0             -19.95293
4  Resort Hotel  98.0               3.04707


In [10]:
pivot = pd.pivot_table(
    df,
    values="adr",
    index="hotel",
    columns="customer_type",
    aggfunc="mean"
)

print("Pivot table:")
print(pivot)

long = pivot.reset_index().melt(
    id_vars="hotel",
    var_name="customer_type",
    value_name="adr"
)

print("\nMelted table:")
print(long)

print("\nRound trip shape:", long.shape)

Pivot table:
customer_type   Contract      Group   Transient  Transient-Party
hotel                                                           
City Hotel     94.054948  91.946416  110.016708        90.873133
Resort Hotel   79.124966  74.762711  101.108248        75.430209

Melted table:
          hotel    customer_type         adr
0    City Hotel         Contract   94.054948
1  Resort Hotel         Contract   79.124966
2    City Hotel            Group   91.946416
3  Resort Hotel            Group   74.762711
4    City Hotel        Transient  110.016708
5  Resort Hotel        Transient  101.108248
6    City Hotel  Transient-Party   90.873133
7  Resort Hotel  Transient-Party   75.430209

Round trip shape: (8, 3)


/tmp/ipykernel_57552/2945415392.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = pd.pivot_table(


In [11]:
left = df[["hotel", "adr"]].drop_duplicates()
right = df[["hotel", "lead_time"]].drop_duplicates()

inner = pd.merge(left, right, on="hotel", how="inner")
left_merge = pd.merge(left, right, on="hotel", how="left")
outer = pd.merge(left, right, on="hotel", how="outer")

print("Inner rows:", len(inner))
print("Left rows:", len(left_merge))
print("Outer rows:", len(outer))

try:
    pd.merge(left, right, on="hotel", how="inner", validate="one_to_one")
    print("Validation passed")
except:
    print("Validation failed: many-to-many or duplicate keys detected")

Inner rows: 4871025
Left rows: 4871025
Outer rows: 4871025
Validation failed: many-to-many or duplicate keys detected


In [12]:
df = df.sort_values(["hotel", "reservation_status_date"])

df["adr_rolling_7"] = (
    df.groupby("hotel")["adr"]
      .transform(lambda x: x.rolling(7).mean())
)

df["adr_rank"] = df.groupby("hotel")["adr"].rank(method="average")

print(df[[
    "hotel",
    "reservation_status_date",
    "adr",
    "adr_rolling_7",
    "adr_rank"
]].head(10))

            hotel reservation_status_date   adr  adr_rolling_7  adr_rank
73714  City Hotel              2014-10-17  62.8            NaN    7843.5
73715  City Hotel              2014-10-17  62.8            NaN    7843.5
73716  City Hotel              2014-10-17  62.8            NaN    7843.5
73717  City Hotel              2014-10-17  62.8            NaN    7843.5
73718  City Hotel              2014-10-17  62.8            NaN    7843.5
73719  City Hotel              2014-10-17  62.8            NaN    7843.5
73720  City Hotel              2014-10-17  62.8           62.8    7843.5
73721  City Hotel              2014-10-17  62.8           62.8    7843.5
73722  City Hotel              2014-10-17  62.8           62.8    7843.5
73723  City Hotel              2014-10-17  62.8           62.8    7843.5


In [13]:
suspicious = df.nlargest(3, "adr")

print(suspicious[[
    "hotel",
    "adr",
    "adults",
    "children",
    "babies",
    "reservation_status"
]])

               hotel     adr  adults  children  babies reservation_status
48515     City Hotel  5400.0       2       0.0       0           Canceled
111403    City Hotel   510.0       1       0.0       0          Check-Out
15083   Resort Hotel   508.0       2       0.0       0          Check-Out


In [14]:
import os
import time

df.to_csv("hotel_cleaned.csv", index=False)
df.to_parquet("hotel_cleaned.parquet", index=False)

csv_size = os.path.getsize("hotel_cleaned.csv") / 1024**2
parquet_size = os.path.getsize("hotel_cleaned.parquet") / 1024**2

start = time.time()
pd.read_csv("hotel_cleaned.csv")
csv_time = time.time() - start

start = time.time()
pd.read_parquet("hotel_cleaned.parquet")
parquet_time = time.time() - start

print("CSV size:", round(csv_size, 2), "MB")
print("Parquet size:", round(parquet_size, 2), "MB")
print("CSV load time:", round(csv_time, 4), "seconds")
print("Parquet load time:", round(parquet_time, 4), "seconds")

CSV size: 22.91 MB
Parquet size: 2.09 MB
CSV load time: 0.569 seconds
Parquet load time: 0.125 seconds
